# GELU alpha-table CUDA test

Compare `torch.nn.functional.gelu`, scalar-alpha optimized GELU, and the new input-dependent alpha-table optimized GELU.

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

In [ ]:
import shutil
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}

GELU_EXT_DIR = PROJECT_DIR / "mrcp_quant" / "optimized_layers" / "gelu"

In [ ]:
# Rebuild after editing gelu_cuda*.{cpp,cu}, trptq_gelu.py, or common/tr_math.cuh.
gelu_dir = Path(GELU_EXT_DIR)
shutil.rmtree(gelu_dir / "build", ignore_errors=True)
for so_path in gelu_dir.glob("_trptq_gelu*.so"):
    so_path.unlink()

!{sys.executable} -m pip install -v --no-build-isolation --no-cache-dir -e "{GELU_EXT_DIR}"

In [ ]:
import importlib
import _trptq_gelu
import trptq_gelu

trptq_gelu = importlib.reload(trptq_gelu)
print("Loaded:", trptq_gelu.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())

for name in ["sigmoid_arith", "sigmoid_mixed_arith", "sigmoid_alpha_arith", "sigmoid_alpha_mixed_arith", "gelu_arith", "gelu_alpha_table_arith"]:
    if not hasattr(_trptq_gelu, name):
        raise RuntimeError(f"Missing CUDA export: {name}. Rebuild and restart the runtime.")
print("GELU exports: OK")


In [ ]:
assert torch.cuda.is_available(), "Select a CUDA runtime first."
device = torch.device("cuda")
torch.manual_seed(0)

lut = trptq_gelu.build_exp_lut_u8(device=device)
alpha_lut = torch.tensor(trptq_gelu.DEFAULT_ALPHA_TABLE, device=device, dtype=torch.float32)

x = torch.randn(1_000_000, device=device, dtype=torch.float32) * 2.0
ref = F.gelu(x)
scalar = trptq_gelu.gelu(x, alpha=1.702, exp_lut=lut)
alpha_table = trptq_gelu.gelu_alpha_table(x, alpha_lut=alpha_lut, exp_lut=lut)

def summarize(name, y):
    err = (y - ref).detach().cpu()
    return {
        "name": name,
        "max_abs": err.abs().max().item(),
        "mean_abs": err.abs().mean().item(),
        "rmse": err.square().mean().sqrt().item(),
    }

rows = [summarize("scalar_alpha", scalar), summarize("alpha_table", alpha_table)]
print(f"{'variant':<16} {'max_abs':>12} {'mean_abs':>12} {'rmse':>12}")
print("-" * 56)
for row in rows:
    print(f"{row['name']:<16} {row['max_abs']:12.4e} {row['mean_abs']:12.4e} {row['rmse']:12.4e}")

In [ ]:
# Test sigmoid(alpha*x) uint8 variants.
sig_alpha = trptq_gelu.sigmoid_alpha_u8(x, alpha=1.702, exp_lut=lut)
sig_alpha_mixed = trptq_gelu.sigmoid_alpha_mixed_u8(x, alpha=1.702, exp_lut=lut)
sig_alpha_ref = torch.round(torch.sigmoid(x * 1.702) * 255.0).clamp(0, 255).to(torch.uint8)

print("sigmoid(alpha*x) uint8 variants vs torch sigmoid")
print(f"{'variant':<14} {'max_abs_q':>10} {'mean_abs_q':>12} {'rmse_float':>12}")
print("-" * 54)
for name, y in [("default", sig_alpha), ("mixed", sig_alpha_mixed)]:
    err = y.to(torch.int32).cpu() - sig_alpha_ref.to(torch.int32).cpu()
    print(
        f"{name:<14} {int(err.abs().max()):10d} "
        f"{err.abs().float().mean().item():12.4f} "
        f"{(err.float() / 255.0).square().mean().sqrt().item():12.4e}"
    )
print("default sample:", sig_alpha[:8].float().cpu() / 255.0)
print("mixed sample:", sig_alpha_mixed[:8].float().cpu() / 255.0)


In [ ]:
x = torch.arange(-128, 128, device=device, dtype=torch.int16)
x_q44 = (1.702*x).clamp_(-128,127).to(torch.int8)
x_q = (x).clamp_(-128,127).to(torch.int8)
fx_q44 = x_q44.float() / 16.0

lut = trptq_gelu.build_exp_lut_u8(device=device)

sig_old = trptq_gelu.sigmoid_u8_from_q44(x_q44, lut)
sig_float = torch.sigmoid(fx_q44)

def benchmark_cuda(fn, warmup=20, repeats=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


torch_ms = benchmark_cuda(lambda: torch.sigmoid(fx_q44))
scalar_ms = benchmark_cuda(lambda: (trptq_gelu.sigmoid_u8_from_q44(x_q44, lut)))

torch.sigmoid(fx_q44)*255, trptq_gelu.sigmoid_u8_from_q44(x_q44, lut)

In [ ]:
print(f"{'kernel':<18} {'ms':>10} {'speedup vs torch':>18}")
print("-" * 50)
print(f"{'torch_F_gelu':<18} {torch_ms:10.4f} {'1.00':>18}x")
print(f"{'scalar_alpha':<18} {scalar_ms:10.4f} {torch_ms / scalar_ms:18.2f}x")


In [ ]:
sig_alpha, sig_alpha_ref, (sig_alpha).int() - (sig_alpha_ref).int()

In [ ]:
def benchmark_cuda(fn, warmup=20, repeats=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


torch_ms = benchmark_cuda(lambda: F.gelu(x))
scalar_ms = benchmark_cuda(lambda: trptq_gelu.gelu(x, alpha=1.702, exp_lut=lut))
table_ms = benchmark_cuda(lambda: trptq_gelu.gelu_alpha_table(x, alpha_lut=alpha_lut, exp_lut=lut))

print(f"{'kernel':<18} {'ms':>10} {'speedup vs torch':>18}")
print("-" * 50)
print(f"{'torch_F_gelu':<18} {torch_ms:10.4f} {'1.00':>18}x")
print(f"{'scalar_alpha':<18} {scalar_ms:10.4f} {torch_ms / scalar_ms:18.2f}x")
print(f"{'alpha_table':<18} {table_ms:10.4f} {torch_ms / table_ms:18.2f}x")

In [ ]:
# Sanity check: if every table entry is 1.702, alpha-table GELU matches scalar-alpha GELU exactly.
flat_alpha = torch.full((10,), 1.702, device=device, dtype=torch.float32)
flat_table = trptq_gelu.gelu_alpha_table(x, alpha_lut=flat_alpha, exp_lut=lut)
torch.testing.assert_close(flat_table, scalar, rtol=0, atol=0)
print("flat alpha table matches scalar-alpha GELU")